# Phase 5 — Regime-Conditioned TFT (Colab GPU)

Three variants of regime-aware TFT are trained and compared against the Phase 4 Colab baseline:

| Variant | Mechanism |
|---|---|
| **TFT-Conditioned** | Single TFT, `regime` as time-varying known categorical |
| **TFT-Separate** | K=4 TFTs, each trained only on sequences ending in its regime |
| **TFT-Ensemble** | Soft blend of K Separate predictions, weighted by HMM posteriors |

**Same capacity as Phase 4 Colab:** `hidden_size=64`, `batch_size=128`, 50 epochs, GPU.

**Known constraint:** Regime 0 has 1572 train samples but **0 val samples** (HMM put no validation days into regime 0).
Its TFT-Separate model is trained but cannot be validated until Phase 6 test set.

## Step 1 — Install Dependencies

In [ ]:
!pip install -q pytorch-forecasting pytorch-lightning lightning hmmlearn scikit-learn

## Step 2 — Mount Drive

Make sure your `regime-forecasting` folder is in Drive, including `data/raw/master.csv`, `results/hmm_model.pkl`, `results/feature_scaler.pkl`, and `results/tft_global_colab_predictions.pkl` (Phase 4 baseline).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/regime-forecasting'
os.chdir(PROJECT_PATH)

for required in ['data/raw/master.csv', 'results/hmm_model.pkl',
                  'results/feature_scaler.pkl', 'results/tft_global_colab_predictions.pkl']:
    assert os.path.exists(required), f'MISSING: {required}'
print('All required artifacts present.')

## Step 3 — Imports and GPU Check

In [ ]:
import sys, pickle, warnings, logging
from pathlib import Path
warnings.filterwarnings('ignore')
logging.getLogger('lightning.pytorch').setLevel(logging.ERROR)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping
from torch.utils.data import Subset, DataLoader
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import MAE

sns.set_theme(style='darkgrid')
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

## Step 4 — Configuration

In [ ]:
TRAIN_END = '2017-12-31'
VAL_END   = '2020-12-31'

RESULTS = Path('results'); FIGURES = Path('results/figures'); METRICS = Path('results/metrics')
for p in (RESULTS, FIGURES, METRICS): p.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'max_encoder_length': 60,
    'max_prediction_length': 1,
    'hidden_size': 64,
    'attention_head_size': 4,
    'dropout': 0.1,
    'hidden_continuous_size': 16,
    'learning_rate': 3e-4,
    'batch_size': 128,
    'max_epochs': 50,
    'patience': 8,
    'gradient_clip_val': 0.1,
}
torch.set_float32_matmul_precision('high')
pl.seed_everything(42, workers=True)
print('Config:', CONFIG)

## Step 5 — Helper Functions (inline)

In [ ]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).flatten(); y_pred = np.asarray(y_pred).flatten()
    mae  = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mask = y_true != 0
    dir_acc = float(np.mean(np.sign(y_true[mask]) == np.sign(y_pred[mask])))
    return {'mae': mae, 'rmse': rmse, 'dir_acc': dir_acc}


def compute_per_regime_metrics(y_true, y_pred, regime_labels):
    out = {}
    for r in np.unique(regime_labels):
        mask = regime_labels == r
        if mask.sum() == 0: continue
        out[int(r)] = {**compute_metrics(np.asarray(y_true)[mask], np.asarray(y_pred)[mask]), 'n': int(mask.sum())}
    return out


def prepare_tft_df(df):
    out = df.copy().sort_index()
    out['time_idx'] = np.arange(len(out), dtype=np.int64)
    out['group_id'] = 'global'
    out['fed_funds_lag1']    = out['fed_funds_rate'].shift(1)
    out['yield_spread_lag1'] = out['yield_spread'].shift(1)
    dow = out.index.dayofweek; month = out.index.month
    out['dow_sin']   = np.sin(2*np.pi*dow/5);   out['dow_cos']   = np.cos(2*np.pi*dow/5)
    out['month_sin'] = np.sin(2*np.pi*month/12); out['month_cos'] = np.cos(2*np.pi*month/12)
    out = out.dropna()
    out['regime'] = out['regime_label'].astype(int).astype(str)
    return out


def build_baseline_dataset(df, training_cutoff):
    """No-regime dataset (used for TFT-Separate, since each model is regime-specialised)."""
    return TimeSeriesDataSet(
        df[df['time_idx'] <= training_cutoff],
        time_idx='time_idx', target='sp500_return', group_ids=['group_id'],
        min_encoder_length=CONFIG['max_encoder_length'], max_encoder_length=CONFIG['max_encoder_length'],
        min_prediction_length=1, max_prediction_length=1,
        static_categoricals=['group_id'],
        time_varying_known_reals=['time_idx','dow_sin','dow_cos','month_sin','month_cos','fed_funds_lag1','yield_spread_lag1'],
        time_varying_unknown_reals=['sp500_return','vix'],
        target_normalizer=GroupNormalizer(groups=['group_id'], transformation=None),
        add_relative_time_idx=True, add_target_scales=True, add_encoder_length=True,
        allow_missing_timesteps=False,
    )


def build_conditioned_dataset(df, training_cutoff):
    """Same as baseline plus `regime` as time-varying known categorical (TFT-Conditioned)."""
    return TimeSeriesDataSet(
        df[df['time_idx'] <= training_cutoff],
        time_idx='time_idx', target='sp500_return', group_ids=['group_id'],
        min_encoder_length=CONFIG['max_encoder_length'], max_encoder_length=CONFIG['max_encoder_length'],
        min_prediction_length=1, max_prediction_length=1,
        static_categoricals=['group_id'],
        time_varying_known_categoricals=['regime'],
        time_varying_known_reals=['time_idx','dow_sin','dow_cos','month_sin','month_cos','fed_funds_lag1','yield_spread_lag1'],
        time_varying_unknown_reals=['sp500_return','vix'],
        target_normalizer=GroupNormalizer(groups=['group_id'], transformation=None),
        add_relative_time_idx=True, add_target_scales=True, add_encoder_length=True,
        allow_missing_timesteps=False,
    )


def build_tft(training_dataset):
    return TemporalFusionTransformer.from_dataset(
        training_dataset,
        learning_rate=CONFIG['learning_rate'],
        hidden_size=CONFIG['hidden_size'],
        attention_head_size=CONFIG['attention_head_size'],
        dropout=CONFIG['dropout'],
        hidden_continuous_size=CONFIG['hidden_continuous_size'],
        loss=MAE(), log_interval=0, optimizer='adam', reduce_on_plateau_patience=4,
    )


def get_regime_indices(dataset, df, target_regime):
    """Indices into dataset where prediction-day's regime equals target_regime.

    pytorch-forecasting's dataset.index has columns: index_start, index_end, sequence_length, time, sequence_id.
    `time` is the encoder-start time_idx; pred-day time_idx = time + sequence_length - max_prediction_length.
    """
    regime_at_t = df.set_index('time_idx')['regime_label'].to_dict()
    idx = dataset.index
    pred_times = (idx['time'] + idx['sequence_length'] - dataset.max_prediction_length).values
    return [i for i, t in enumerate(pred_times) if regime_at_t.get(int(t)) == target_regime]


def make_trainer(name):
    return pl.Trainer(
        max_epochs=CONFIG['max_epochs'], accelerator='gpu', devices=1,
        gradient_clip_val=CONFIG['gradient_clip_val'],
        callbacks=[EarlyStopping(monitor='val_loss', patience=CONFIG['patience'], mode='min', verbose=True)],
        enable_progress_bar=True, enable_model_summary=False, logger=False, num_sanity_val_steps=0,
    )

print('Helpers defined.')

## Step 6 — Load Data, HMM, Phase 4 Baseline

In [ ]:
df = pd.read_csv('data/raw/master.csv', index_col=0, parse_dates=True).sort_index()
df_phase5 = df.loc[:VAL_END].copy()
val_mask  = df_phase5.index > TRAIN_END
y_val_actual_full = df_phase5.loc[val_mask, 'sp500_return'].values
val_dates_full    = df_phase5.index[val_mask]
regime_val_full   = df_phase5.loc[val_mask, 'regime_label'].values

with open('results/hmm_model.pkl', 'rb') as f: hmm_model = pickle.load(f)
with open('results/feature_scaler.pkl', 'rb') as f: scaler = pickle.load(f)

with open('results/tft_global_colab_predictions.pkl', 'rb') as f: phase4 = pickle.load(f)
phase4_metrics = phase4['metrics']

print(f'Phase 5 rows: {len(df_phase5):,}  |  Val rows: {val_mask.sum()}')
print(f'HMM K={hmm_model.n_components}  |  Phase 4 baseline DirAcc={phase4_metrics["dir_acc"]*100:.2f}%')
print()
print('Train regime counts:', dict(pd.Series(df_phase5.loc[~val_mask,"regime_label"]).value_counts().sort_index()))
print('Val   regime counts:', dict(pd.Series(regime_val_full).value_counts().sort_index()))

## Step 7 — Prepare TFT DataFrame and Datasets

In [ ]:
tft_df = prepare_tft_df(df_phase5)
training_cutoff = tft_df.loc[tft_df.index <= TRAIN_END, 'time_idx'].max()
print(f'tft_df shape: {tft_df.shape}  |  training_cutoff: {training_cutoff}')

# Baseline (no-regime) datasets for TFT-Separate
base_train_ds = build_baseline_dataset(tft_df, training_cutoff)
base_val_ds   = TimeSeriesDataSet.from_dataset(base_train_ds, tft_df, predict=False, stop_randomization=True, min_prediction_idx=training_cutoff+1)

# Conditioned datasets (regime as input)
cond_train_ds = build_conditioned_dataset(tft_df, training_cutoff)
cond_val_ds   = TimeSeriesDataSet.from_dataset(cond_train_ds, tft_df, predict=False, stop_randomization=True, min_prediction_idx=training_cutoff+1)

print(f'Baseline    train: {len(base_train_ds):,}  val: {len(base_val_ds):,}')
print(f'Conditioned train: {len(cond_train_ds):,}  val: {len(cond_val_ds):,}')

## Step 8 — Train TFT-Conditioned

Single model, regime is input. This is the cleanest test of "does regime info help?"

In [ ]:
cond_train_dl = cond_train_ds.to_dataloader(train=True, batch_size=CONFIG['batch_size'], num_workers=4, persistent_workers=True)
cond_val_dl   = cond_val_ds.to_dataloader(train=False, batch_size=256, num_workers=4, persistent_workers=True)

cond_model = build_tft(cond_train_ds)
print('Params:', sum(p.numel() for p in cond_model.parameters()))

trainer = make_trainer('conditioned')
print('\nTraining TFT-Conditioned...')
trainer.fit(cond_model, train_dataloaders=cond_train_dl, val_dataloaders=cond_val_dl)
cond_stopped = trainer.current_epoch
print(f'Stopped at epoch {cond_stopped}')

raw = cond_model.predict(cond_val_dl, mode='prediction', return_x=False,
                          trainer_kwargs={'accelerator':'gpu','devices':1,'logger':False,'enable_progress_bar':False})
y_pred_cond = raw.cpu().numpy().flatten()
n = len(y_pred_cond)
y_actual_cond = y_val_actual_full[-n:]
regime_cond   = regime_val_full[-n:]
dates_cond    = val_dates_full[-n:]

metrics_cond  = compute_metrics(y_actual_cond, y_pred_cond)
perreg_cond   = compute_per_regime_metrics(y_actual_cond, y_pred_cond, regime_cond)
print(f'\nTFT-Conditioned: MAE={metrics_cond["mae"]:.5f}  RMSE={metrics_cond["rmse"]:.5f}  DirAcc={metrics_cond["dir_acc"]*100:.2f}%')
for k,v in perreg_cond.items():
    print(f'  Regime {k} (n={v["n"]:3d}): MAE={v["mae"]:.5f}  DirAcc={v["dir_acc"]*100:.2f}%')

## Step 9 — Train TFT-Separate (K=4)

One model per regime. Each sees only training sequences whose prediction day is in that regime.
Regime 0 has zero val samples — early stopping on regime 0 model uses train loss only (skipped val).

In [ ]:
K = hmm_model.n_components
regime_models = {}
regime_train_idx = {k: get_regime_indices(base_train_ds, tft_df, k) for k in range(K)}
regime_val_idx   = {k: get_regime_indices(base_val_ds,   tft_df, k) for k in range(K)}

for k in range(K):
    print(f'\nRegime {k}: train={len(regime_train_idx[k])}  val={len(regime_val_idx[k])}')

for k in range(K):
    n_train, n_val = len(regime_train_idx[k]), len(regime_val_idx[k])
    if n_train == 0:
        print(f'\n[Regime {k}] no train samples — skipping')
        continue
    print(f'\n=== Training TFT-Separate model for regime {k} (train={n_train}, val={n_val}) ===')

    train_subset = Subset(base_train_ds, regime_train_idx[k])
    train_dl = DataLoader(train_subset, batch_size=CONFIG['batch_size'], shuffle=True,
                          num_workers=4, persistent_workers=True, collate_fn=base_train_ds._collate_fn)

    if n_val > 0:
        val_subset = Subset(base_val_ds, regime_val_idx[k])
        val_dl = DataLoader(val_subset, batch_size=256, shuffle=False,
                            num_workers=4, persistent_workers=True, collate_fn=base_val_ds._collate_fn)
        callbacks = [EarlyStopping(monitor='val_loss', patience=CONFIG['patience'], mode='min', verbose=True)]
    else:
        val_dl = None
        callbacks = []  # No early stopping without val
        print(f'  (no val samples — running full {CONFIG["max_epochs"]} epochs without early stopping)')

    model_k = build_tft(base_train_ds)
    trainer_k = pl.Trainer(
        max_epochs=CONFIG['max_epochs'], accelerator='gpu', devices=1,
        gradient_clip_val=CONFIG['gradient_clip_val'], callbacks=callbacks,
        enable_progress_bar=True, enable_model_summary=False, logger=False, num_sanity_val_steps=0,
    )
    trainer_k.fit(model_k, train_dataloaders=train_dl, val_dataloaders=val_dl)
    regime_models[k] = model_k
    print(f'  done @ epoch {trainer_k.current_epoch}')

## Step 10 — Validate TFT-Separate on Val Set

In [ ]:
# pytorch-forecasting's model.predict() asserts dataloader.dataset is a TimeSeriesDataSet,
# which Subset breaks. Workaround: predict on the FULL val dataloader, then mask by regime.
# Bonus — we cache all_preds here for Step 11 (Ensemble) to reuse, avoiding 4x duplicate predictions.

full_val_dl = base_val_ds.to_dataloader(train=False, batch_size=256, num_workers=4, persistent_workers=True)

# Map each val sample index to the regime of its prediction day
val_pred_times = (base_val_ds.index['time'] + base_val_ds.index['sequence_length'] - base_val_ds.max_prediction_length).values
regime_at_t = tft_df.set_index('time_idx')['regime_label'].to_dict()
regime_at_pred = np.array([int(regime_at_t.get(int(t), -1)) for t in val_pred_times])

all_preds = {}
y_pred_sep_full = np.full(len(base_val_ds), np.nan)
for k in range(K):
    if k not in regime_models: continue
    print(f'Predicting with regime-{k} model on full val ({len(base_val_ds)} samples)...')
    raw = regime_models[k].predict(
        full_val_dl, mode='prediction', return_x=False,
        trainer_kwargs={'accelerator':'gpu','devices':1,'logger':False,'enable_progress_bar':False}
    )
    all_preds[k] = raw.cpu().numpy().flatten()
    mask = regime_at_pred == k
    if mask.sum() > 0:
        y_pred_sep_full[mask] = all_preds[k][mask]
    else:
        print(f'  Regime {k}: no val samples — predictions cached for Ensemble only (Separate eval deferred to Phase 6)')

# Align with actuals (drop the leading rows TFT can't predict due to encoder_length)
n_pred_sep = len(base_val_ds)
y_actual_sep = y_val_actual_full[-n_pred_sep:]
regime_sep   = regime_val_full[-n_pred_sep:]
dates_sep    = val_dates_full[-n_pred_sep:]

valid_mask = ~np.isnan(y_pred_sep_full)
metrics_sep = compute_metrics(y_actual_sep[valid_mask], y_pred_sep_full[valid_mask])
perreg_sep  = compute_per_regime_metrics(y_actual_sep[valid_mask], y_pred_sep_full[valid_mask], regime_sep[valid_mask])
print(f'\nTFT-Separate: MAE={metrics_sep["mae"]:.5f}  RMSE={metrics_sep["rmse"]:.5f}  DirAcc={metrics_sep["dir_acc"]*100:.2f}%  n={valid_mask.sum()}')
for k,v in perreg_sep.items():
    print(f'  Regime {k} (n={v["n"]:3d}): MAE={v["mae"]:.5f}  DirAcc={v["dir_acc"]*100:.2f}%')

## Step 11 — TFT-Ensemble (Soft Blend Using HMM Posteriors)

For each val day:
- Compute HMM posteriors P(regime=k | features at that day) using the FROZEN train-only HMM
- Run all K Separate models on every val day
- Final pred = sum_k posterior_k * pred_k

In [ ]:
# Reuse all_preds cached by Step 10 (one full-val pass per regime model — already done)
n_pred_ens = len(next(iter(all_preds.values())))
print(f'Using cached all_preds from Step 10 ({len(all_preds)} models, {n_pred_ens} val days each)')

# Compute HMM posteriors on val days (no refitting — frozen train-only HMM)
feats_val = df_phase5.loc[val_dates_full, ['realized_vol_20d','yield_spread','vix']].values
X_val = scaler.transform(feats_val)
posteriors_full = hmm_model.predict_proba(X_val)
posteriors = posteriors_full[-n_pred_ens:]
print(f'Posteriors shape: {posteriors.shape}  (val_days x K)')

# Soft-blend: pred_t = sum_k P(regime=k|features_t) * model_k(t)
y_pred_ens = np.zeros(n_pred_ens)
for k in range(K):
    if k in all_preds:
        y_pred_ens += posteriors[:, k] * all_preds[k]

y_actual_ens = y_val_actual_full[-n_pred_ens:]
regime_ens   = regime_val_full[-n_pred_ens:]
dates_ens    = val_dates_full[-n_pred_ens:]
metrics_ens  = compute_metrics(y_actual_ens, y_pred_ens)
perreg_ens   = compute_per_regime_metrics(y_actual_ens, y_pred_ens, regime_ens)
print(f'\nTFT-Ensemble: MAE={metrics_ens["mae"]:.5f}  RMSE={metrics_ens["rmse"]:.5f}  DirAcc={metrics_ens["dir_acc"]*100:.2f}%')
for k,v in perreg_ens.items():
    print(f'  Regime {k} (n={v["n"]:3d}): MAE={v["mae"]:.5f}  DirAcc={v["dir_acc"]*100:.2f}%')

## Step 12 — Comparison Table (vs Phase 4 Colab Baseline)

In [ ]:
rows = [
    {'model': 'Phase 4 — TFT Global (baseline)', **phase4_metrics},
    {'model': 'Phase 5 — TFT-Conditioned',       **metrics_cond},
    {'model': 'Phase 5 — TFT-Separate',          **metrics_sep},
    {'model': 'Phase 5 — TFT-Ensemble',          **metrics_ens},
]
comp = pd.DataFrame(rows)
comp['dir_acc_pct'] = (comp['dir_acc']*100).round(2)
print(comp[['model','mae','rmse','dir_acc_pct']].to_string(index=False))
comp.to_csv('results/metrics/phase5_comparison.csv', index=False)

# Per-regime breakdown
rows_pr = []
for variant_name, perreg in [('Conditioned', perreg_cond), ('Separate', perreg_sep), ('Ensemble', perreg_ens)]:
    for k, m in perreg.items():
        rows_pr.append({'variant': variant_name, 'regime': k, 'n': m['n'],
                         'mae': m['mae'], 'rmse': m['rmse'], 'dir_acc_pct': m['dir_acc']*100})
perreg_df = pd.DataFrame(rows_pr)
print('\nPer-regime metrics:')
print(perreg_df.to_string(index=False))
perreg_df.to_csv('results/metrics/phase5_per_regime.csv', index=False)
print('\nSaved phase5_comparison.csv and phase5_per_regime.csv')

## Step 13 — Gate Checks

In [ ]:
for name, m in [('Conditioned', metrics_cond), ('Separate', metrics_sep), ('Ensemble', metrics_ens)]:
    assert 0.40 < m['dir_acc'] < 0.60, f'FAIL {name}: DirAcc={m["dir_acc"]:.3f} outside [40%, 60%] — possible lookahead'
    assert 0.001 < m['mae'] < 0.05,    f'FAIL {name}: MAE={m["mae"]:.5f} implausible'
print('All Phase 5 gate checks PASSED')
print()
print('Hypothesis check: did regime conditioning improve DirAcc over the global baseline?')
for name, m in [('Conditioned', metrics_cond), ('Separate', metrics_sep), ('Ensemble', metrics_ens)]:
    delta = (m['dir_acc'] - phase4_metrics['dir_acc'])*100
    arrow = '+' if delta > 0 else ''
    print(f'  {name:11s}: {arrow}{delta:.2f}pp vs Phase 4 baseline')

## Step 14 — Visualisation

In [ ]:
REGIME_COLORS = {0:'#4CAF50', 1:'#F44336', 2:'#FF9800', 3:'#2196F3'}

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle('Phase 5 — Regime-Conditioned TFT vs Global Baseline', fontsize=13)

# DirAcc bar chart
ax = axes[0,0]
labels = ['Phase 4\nGlobal', 'Conditioned', 'Separate', 'Ensemble']
vals   = [phase4_metrics['dir_acc']*100, metrics_cond['dir_acc']*100, metrics_sep['dir_acc']*100, metrics_ens['dir_acc']*100]
colors = ['gray', 'steelblue', 'darkorange', 'mediumseagreen']
bars = ax.bar(labels, vals, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(50, color='gray', linestyle='--', linewidth=0.8, label='Random')
ax.axhline(60, color='red',  linestyle='--', linewidth=0.8, label='Lookahead alert')
ax.set_title('Directional Accuracy (%)'); ax.set_ylim(40, 65); ax.legend(fontsize=8)
for b, v in zip(bars, vals): ax.text(b.get_x()+b.get_width()/2, v+0.3, f'{v:.2f}%', ha='center', fontsize=9)

# MAE bar chart
ax = axes[0,1]
vals = [phase4_metrics['mae'], metrics_cond['mae'], metrics_sep['mae'], metrics_ens['mae']]
bars = ax.bar(labels, vals, color=colors, edgecolor='black', linewidth=0.5)
ax.set_title('MAE (lower better)')
for b, v in zip(bars, vals): ax.text(b.get_x()+b.get_width()/2, v*1.001, f'{v:.5f}', ha='center', fontsize=8)

# Per-regime DirAcc heat: rows = variants, cols = regimes
ax = axes[1,0]
regimes_present = sorted(set().union(perreg_cond.keys(), perreg_sep.keys(), perreg_ens.keys()))
matrix = []
for perreg in [perreg_cond, perreg_sep, perreg_ens]:
    matrix.append([perreg.get(r, {'dir_acc': np.nan})['dir_acc']*100 for r in regimes_present])
sns.heatmap(matrix, annot=True, fmt='.1f', cmap='RdYlGn', center=50, vmin=40, vmax=60,
             xticklabels=[f'Reg {r}' for r in regimes_present],
             yticklabels=['Conditioned','Separate','Ensemble'], ax=ax, cbar_kws={'label':'DirAcc %'})
ax.set_title('Per-Regime Directional Accuracy')

# Predictions overlay (zoomed on COVID Q1 2020)
ax = axes[1,1]
covid = (dates_ens >= '2020-02-01') & (dates_ens <= '2020-04-30')
ax.plot(dates_ens[covid], y_actual_ens[covid], color='black', label='Actual', linewidth=1.0, marker='o', markersize=3)
ax.plot(dates_ens[covid], y_pred_ens[covid],   color='mediumseagreen', label='Ensemble', linewidth=1.0)
ax.plot(dates_cond[covid], y_pred_cond[covid], color='steelblue',     label='Conditioned', linewidth=1.0, alpha=0.7)
ax.set_title('COVID Crash Q1 2020 — Predictions vs Actual'); ax.legend(fontsize=8); ax.axhline(0, color='gray', linewidth=0.4, linestyle='--')

plt.tight_layout()
plt.savefig('results/figures/viz_5_regime_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/viz_5_regime_comparison.png')

## Step 15 — Save Artifacts

In [ ]:
torch.save(cond_model.state_dict(), 'results/tft_conditioned_colab.ckpt')
for k, m in regime_models.items():
    torch.save(m.state_dict(), f'results/tft_separate_regime{k}_colab.ckpt')

with open('results/tft_conditioned_colab_predictions.pkl', 'wb') as f:
    pickle.dump({'val_dates': dates_cond, 'y_actual': y_actual_cond, 'y_pred': y_pred_cond,
                  'regime': regime_cond, 'metrics': metrics_cond, 'per_regime': perreg_cond,
                  'config': CONFIG, 'stopped_epoch': cond_stopped}, f)
with open('results/tft_separate_colab_predictions.pkl', 'wb') as f:
    pickle.dump({'val_dates': dates_sep, 'y_actual': y_actual_sep, 'y_pred': y_pred_sep_full,
                  'regime': regime_sep, 'metrics': metrics_sep, 'per_regime': perreg_sep,
                  'config': CONFIG, 'regime_train_counts': {k: len(v) for k,v in regime_train_idx.items()}}, f)
with open('results/tft_ensemble_colab_predictions.pkl', 'wb') as f:
    pickle.dump({'val_dates': dates_ens, 'y_actual': y_actual_ens, 'y_pred': y_pred_ens,
                  'regime': regime_ens, 'metrics': metrics_ens, 'per_regime': perreg_ens,
                  'posteriors': posteriors, 'all_preds': all_preds, 'config': CONFIG}, f)
print('Saved checkpoints and prediction pkls.')

## Step 16 — Download to Mac

In [ ]:
from google.colab import files
for fname in [
    'results/tft_conditioned_colab.ckpt',
    'results/tft_conditioned_colab_predictions.pkl',
    'results/tft_separate_colab_predictions.pkl',
    'results/tft_ensemble_colab_predictions.pkl',
    'results/metrics/phase5_comparison.csv',
    'results/metrics/phase5_per_regime.csv',
    'results/figures/viz_5_regime_comparison.png',
]:
    try: files.download(fname)
    except Exception as e: print(f'Skip {fname}: {e}')
for k in range(4):
    p = f'results/tft_separate_regime{k}_colab.ckpt'
    try: files.download(p)
    except: pass
print('Downloads triggered.')